# Turkish Morph Retrieval — 600-family final evaluation

Bu notebook yalnız dondurulmuş **100 development + 500 final/sealed test** benchmark'ı içindir.
Her family: `1 query + 1 positive + 8 hard negative + 2 easy negative`.
Development model/ayar seçimi, 500 final family ise kararlar dondurulduktan sonraki paper sonucu içindir.

V3.6 değerlendirme katmanları: veri/artefakt kontrolü → ucuz baseline'lar → dense encoder'lar → hard-negative
ayrımı → bootstrap CI → paired testler → fenomen/slice analizi → ablation → hata analizi → export.
Kontrollü ve full-corpus retrieval sonuçları ayrı cutoff setleriyle raporlanır.

## 0. Colab kullanımı

1. `Runtime > Change runtime type` ile GPU seçin (Qwen3-8B için A100 önerilir).
2. Aşağıdaki hücreleri sırayla çalıştırın.
3. Beş model A100'de sırayla yüklenir; her modelden sonra GPU belleği temizlenir.
4. A100 dışındaki runtime'larda Qwen3-8B'yi model tablosunda `enabled=False` yapabilirsiniz.
5. Çalışma yarıda kesilirse sonuç cache'i sayesinde tamamlanan modeller yeniden koşmaz.

In [ ]:
%pip -q install -U "sentence-transformers>=3.0,<6" "transformers>=4.48,<5" scikit-learn pandas matplotlib seaborn

In [ ]:
import gc, hashlib, json, random, shutil, subprocess, sys, time, traceback
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from IPython.display import display
pd.options.display.float_format = "{:.3f}".format

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

REPO_URL = "https://github.com/Bur8300/turkish-morph-retrieval.git"
ROOT = Path("/content/turkish-morph-retrieval")

def run_git(command):
    result = subprocess.run(command, text=True, capture_output=True)
    if result.returncode != 0:
        raise RuntimeError("Public GitHub deposuna erisilemedi. Runtime internet baglantisini ve REPO_URL degerini kontrol edin.\nGit: " + result.stderr[-800:])

if (ROOT / "test/evaluation.py").exists():
    run_git(["git", "-C", str(ROOT), "pull", "--ff-only"])
else:
    if ROOT.exists(): shutil.rmtree(ROOT)  # yalnız yarım kalmış Colab clone'u
    run_git(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)])
sys.path.insert(0, str(ROOT))

from sentence_transformers import SentenceTransformer
from test.evaluation import (
    EVALUATION_API_VERSION, FULL_CORPUS_RECALL_KS, ablate_items, approximate_randomization,
    bootstrap_ci, candidate_only_classifier, closed_qrels, evaluate_artifacts, evaluate_run,
    holm_adjust, load_items, mcnemar, paired_bootstrap, score_encoder, slice_summary,
)
assert EVALUATION_API_VERSION == "3.2", "Repo eski. Son dosyaları pull edip runtime'ı yeniden başlatın."
GIT_COMMIT = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip()
print("repo:", ROOT)
print("git commit:", GIT_COMMIT)
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 1. Final deney ayarları

Bu notebook preview kabul etmez. Ayarlar development üzerinde dondurulduktan sonra 500-family
final test yalnız nihai rapor için çalıştırılmalıdır.

In [ ]:
RUN_ID = "test_v39_final"
DEV_FILE = ROOT / f"test/runs/{RUN_ID}/release/morph_dev_v3.9.0.json"
TEST_FILE = ROOT / f"test/runs/{RUN_ID}/private/morph_test_internal_v3.9.0.json"
BATCH_SIZE = 16
FULL_RUN_DEPTH = 50             # Full-corpus Recall@50 için
N_BOOT = 10_000
USE_CACHE = True
OUTPUT_NAME = "morph_eval_600_final_v390"
OUTPUT_DIR = Path("/content") / OUTPUT_NAME if Path("/content").exists() else ROOT / "test/results" / OUTPUT_NAME
CACHE_DIR = OUTPUT_DIR / "cache"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def resolve_eval_file(expected):
    if expected.exists():
        return expected
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError(f"Eksik evaluation dosyası: {expected}") from exc
    print(f"Yükleyin: {expected.name}")
    uploaded = files.upload()
    if expected.name not in uploaded:
        raise FileNotFoundError(f"{expected.name} yüklenmedi")
    target = Path("/content") / expected.name
    target.write_bytes(uploaded[expected.name])
    return target

DEV_FILE = resolve_eval_file(DEV_FILE)
TEST_FILE = resolve_eval_file(TEST_FILE)
DATA_FILE = TEST_FILE
DEV = load_items(DEV_FILE)
EVAL_ITEMS = load_items(TEST_FILE)
DATA_NOTICE = "FROZEN: 100 development / 500 sealed protokolü."
assert len(DEV) == 100, f"100 development bekleniyordu, {len(DEV)} bulundu"
assert len(EVAL_ITEMS) == 500, f"500 sealed test bekleniyordu, {len(EVAL_ITEMS)} bulundu"

DATA_SHA256 = hashlib.sha256(DATA_FILE.read_bytes()).hexdigest()
print(DATA_NOTICE)
print(f"family={len(EVAL_ITEMS)} | candidate={sum(len(x['candidates']) for x in EVAL_ITEMS)}")
print("data sha256:", DATA_SHA256)

## 2. Veri bütünlüğü ve dağılım

In [ ]:
integrity_rows = []
for item in EVAL_ITEMS:
    roles = Counter(c["role"] for c in item["candidates"])
    ids = [c["id"] for c in item["candidates"]]
    integrity_rows.append({
        "family_id": item["family_id"],
        "candidate_n": len(ids),
        "positive_n": roles["positive"],
        "hard_n": roles["hard_negative"],
        "easy_n": roles["easy_negative"],
        "unique_ids": len(ids) == len(set(ids)),
        "gold_exists": item["gold_id"] in ids,
    })
integrity = pd.DataFrame(integrity_rows)
display(integrity)
assert integrity[["unique_ids", "gold_exists"]].all().all()
assert (integrity[["candidate_n", "positive_n", "hard_n", "easy_n"]] == [11, 1, 8, 2]).all().all()
family_ids = [item["family_id"] for item in EVAL_ITEMS]
corpus_ids = [candidate["id"] for item in EVAL_ITEMS for candidate in item["candidates"]]
assert len(family_ids) == len(set(family_ids)), "Tekrarlanan family_id var"
assert len(corpus_ids) == len(set(corpus_ids)), "Family'ler arasında tekrarlanan candidate id var"
assert sum(bool(item.get("strict_minimal_pair")) for item in EVAL_ITEMS) == 125
assert Counter(item.get("generator_id") for item in EVAL_ITEMS) == {"generator_a": 250, "generator_b": 250}
print("✓ 1/8/2, 125 strict minimal pair ve 250+250 generator dengesi doğrulandı.")

In [ ]:
fields = [
    "split", "query_sentence_count", "passage_sentence_count", "layer", "objective",
    "generalization_bucket", "macro_phenomenon", "target_feature",
    "strict_minimal_pair", "generator_id",
]
for field in fields:
    values = [item.get("split", item.get("target_split")) if field == "split" else item.get(field) for item in EVAL_ITEMS]
    counts = pd.Series([str(value) for value in values]).value_counts().rename("n").to_frame()
    counts["ratio"] = counts["n"] / len(EVAL_ITEMS)
    print("\n", field)
    display(counts)

## 3. Metrikler nasıl okunmalı?

**Ana morfoloji metrikleri:**

- `pairwise_hard_accuracy`: gold–hard çiftlerinin ne kadarında gold daha yüksek skor aldı? Chance %50.
- `pairwise_morph_hard_accuracy` / `pairwise_semantic_hard_accuracy`: iki zorluk kaynağını ayırır.
- `all_hard_family_consistency`: gold aynı family'deki sekiz hard'ın tamamını geçti mi?
- `contrast_consistency`: gold, minimal morfolojik negatifi geçti mi?
- `hardest_hard_margin`: gold skoru − en yüksek hard skoru. Pozitif değer iyi.

**11 adaylık kontrollü metrikler:** `Recall@1/3`, `MRR@10`, `nDCG@10`.
**5.500-belge full-corpus metrikleri:** `Recall@1/3/10/50`, `MRR@10`, `nDCG@10`.

## 4. Query-blind / ucuz artefakt baseline'ları

In [ ]:
qrels = closed_qrels(EVAL_ITEMS)
artifact_summaries = evaluate_artifacts(DEV, EVAL_ITEMS)
try:
    artifact_summaries["candidate_only_char_tfidf"] = candidate_only_classifier(DEV, EVAL_ITEMS)["summary"]
except Exception as exc:
    print("candidate-only classifier atlandı:", exc)

ARTIFACT_DF = pd.DataFrame(artifact_summaries).T.sort_values("recall@1", ascending=False)
display(ARTIFACT_DF[["recall@1", "recall@3", "mrr@10", "ndcg@10", "mean_rank"]].round(3))
print("closed R@1 chance =", round(1 / 11, 4))

In [ ]:
ax = ARTIFACT_DF["recall@1"].sort_values().plot.barh(figsize=(8, 4), color="#7a9cc6")
ax.axvline(1 / 11, color="black", linestyle="--", label="chance 1/11")
ax.set(title="Ucuz baseline Recall@1", xlabel="Recall@1", ylabel="")
ax.legend(); plt.tight_layout(); plt.show()

## 5. Encoder kayıt defteri

Varsayılanlar A100 icin mevcut proje listesindeki 5 encoder'dir. Bir model yüklenemezse deney durmaz; hata kaydedilir.
Prefix'ler model ailesinin retrieval biçimine göre ayrı tutulur. Qwen3-8B belleği yüksek olduğu
için A100/L4 disi runtime'larda kapatabilirsiniz. Aynı notebook'u tekrar çalıştırırken model veya prefix değişirse cache anahtarı da değişir.

In [ ]:
INSTRUCT = "Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery:"
MODEL_SPECS = [
    {"name": "e5-large", "repo": "intfloat/multilingual-e5-large", "query_prefix": "query: ", "document_prefix": "passage: ", "enabled": True},
    {"name": "bge-m3", "repo": "BAAI/bge-m3", "query_prefix": "", "document_prefix": "", "enabled": True},
    {"name": "modernbert-tr", "repo": "ytu-ce-cosmos/modernbert-tr-embed", "query_prefix": INSTRUCT, "document_prefix": "", "enabled": True},
    {"name": "trmteb-ft-110m", "repo": "trmteb/turkish-embedding-model-fine-tuned", "query_prefix": "", "document_prefix": "", "enabled": True},
    {"name": "qwen3-8b", "repo": "Qwen/Qwen3-Embedding-8B", "query_prefix": INSTRUCT, "document_prefix": "", "enabled": True, "dtype": "float16"},
]
display(pd.DataFrame(MODEL_SPECS)[["name", "repo", "enabled", "query_prefix", "document_prefix"]])

In [ ]:
EVAL_CODE_SHA256 = hashlib.sha256((ROOT / "test/evaluation.py").read_bytes()).hexdigest()

def cache_path(spec):
    payload = json.dumps({"spec": spec, "data": DATA_SHA256, "eval": EVAL_CODE_SHA256}, sort_keys=True)
    key = hashlib.sha256(payload.encode()).hexdigest()[:12]
    return CACHE_DIR / f"{spec['name']}-{key}.json"

def load_model(spec):
    kwargs = {"trust_remote_code": True}
    if spec.get("dtype") and torch.cuda.is_available():
        kwargs["model_kwargs"] = {"torch_dtype": getattr(torch, spec["dtype"])}
    model = SentenceTransformer(spec["repo"], **kwargs)
    try:
        model.default_prompt_name = None
    except Exception:
        pass
    return model

def score_with_backoff(model, items, spec, include_full_run=False):
    batch_size = BATCH_SIZE
    while True:
        try:
            result = score_encoder(
                model, items, spec.get("query_prefix", ""), spec.get("document_prefix", ""),
                full_corpus=False, batch_size=batch_size, include_full_run=include_full_run,
                full_run_depth=FULL_RUN_DEPTH if include_full_run else None,
            )
            return result, batch_size
        except torch.cuda.OutOfMemoryError:
            if batch_size == 1:
                raise
            batch_size = max(1, batch_size // 2)
            gc.collect()
            torch.cuda.empty_cache()
            print(f"CUDA OOM; batch_size={batch_size} ile yeniden deneniyor")

def run_one_model(spec):
    path = cache_path(spec)
    if USE_CACHE and path.exists():
        print(spec["name"], "cache'ten yüklendi")
        return json.loads(path.read_text()), {"model": spec["name"], "cached": True}
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    model = load_model(spec)
    load_seconds = time.perf_counter() - started
    dimension = model.get_sentence_embedding_dimension()
    max_seq_length = model.max_seq_length
    try:
        resolved_revision = model[0].auto_model.config._commit_hash
    except Exception:
        resolved_revision = None
    score_started = time.perf_counter()
    result, effective_batch_size = score_with_backoff(model, EVAL_ITEMS, spec, include_full_run=True)
    score_seconds = time.perf_counter() - score_started
    peak_gb = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0
    path.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    del model
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return result, {
        "model": spec["name"], "cached": False, "dimension": dimension,
        "max_seq_length": max_seq_length, "resolved_revision": resolved_revision,
        "effective_batch_size": effective_batch_size, "load_seconds": load_seconds,
        "score_seconds": score_seconds, "peak_gpu_gb": peak_gb,
    }

## 6. Encoder'ları çalıştır

In [ ]:
RESULTS, RUNTIME_ROWS, MODEL_ERRORS = {}, [], []
for spec in [spec for spec in MODEL_SPECS if spec["enabled"]]:
    print("\n===", spec["name"], "===")
    try:
        RESULTS[spec["name"]], runtime = run_one_model(spec)
        RUNTIME_ROWS.append(runtime)
    except Exception as exc:
        MODEL_ERRORS.append({"model": spec["name"], "error": repr(exc), "traceback": traceback.format_exc()})
        print("ATLANDI:", repr(exc))
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
assert RESULTS, "Hiçbir model tamamlanmadı. MODEL_ERRORS çıktısını inceleyin."
display(pd.DataFrame(RUNTIME_ROWS))
if MODEL_ERRORS: display(pd.DataFrame(MODEL_ERRORS)[["model", "error"]])

## 7. Ana sonuç tablosu

In [ ]:
SUMMARY_DF = pd.DataFrame({name: result["summary"] for name, result in RESULTS.items()}).T
primary = [
    "hard_only_mrr@10", "pairwise_hard_accuracy",
    "pairwise_morph_hard_accuracy", "pairwise_semantic_hard_accuracy",
    "all_hard_family_consistency", "contrast_consistency",
    "hardest_hard_margin", "recall@1", "recall@3", "mrr@10", "ndcg@10",
    "mean_rank", "mean_hard_rank",
]
display(SUMMARY_DF[primary].sort_values("recall@1", ascending=False).round(3))
print("hard-only R@1 chance:", round(1 / 9, 4), "| pairwise chance: 0.5 | closed R@1 chance:", round(1 / 11, 4))

## 8. Query-level bootstrap %95 güven aralıkları

In [ ]:
CI_METRICS = ["pairwise_hard_accuracy", "pairwise_morph_hard_accuracy", "pairwise_semantic_hard_accuracy", "all_hard_family_consistency", "recall@1", "mrr@10", "ndcg@10"]
ci_rows = []
for model_name, result in RESULTS.items():
    for metric in CI_METRICS:
        values = [row[metric] for row in result["per_query"] if row.get(metric) is not None]
        low, high = bootstrap_ci(values, n_boot=N_BOOT, seed=SEED)
        ci_rows.append({"model": model_name, "metric": metric, "mean": np.mean(values), "ci_low": low, "ci_high": high, "n": len(values)})
CI_DF = pd.DataFrame(ci_rows)
display(CI_DF.round(3))
print("Final test CI'ları 500 query üzerinde query-level bootstrap ile hesaplandı.")

In [ ]:
plot_df = CI_DF[CI_DF.metric.isin(["pairwise_hard_accuracy", "recall@1"])].copy()
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, metric in zip(axes, plot_df.metric.unique()):
    part = plot_df[plot_df.metric == metric].sort_values("mean")
    ax.errorbar(part["mean"], part["model"], xerr=[part["mean"] - part["ci_low"], part["ci_high"] - part["mean"]], fmt="o", capsize=3)
    ax.set_title(metric); ax.set_xlim(-0.03, 1.03); ax.grid(axis="x", alpha=.25)
plt.tight_layout(); plt.show()

## 9. Hard-negative subtype analizi

In [ ]:
subtype_rows = []
for model_name, result in RESULTS.items():
    for item in EVAL_ITEMS:
        scores = result["scores"][item["family_id"]]
        gold_score = scores[item["gold_id"]]
        for candidate in item["candidates"]:
            if candidate["role"] != "hard_negative": continue
            margin = gold_score - scores[candidate["id"]]
            subtype_rows.append({
                "model": model_name, "family_id": item["family_id"], "subtype": candidate["subtype"],
                "target_feature": item["target_feature"], "margin": margin,
                "gold_wins": 1.0 if margin > 0 else 0.5 if margin == 0 else 0.0,
            })
SUBTYPE_PAIRS_DF = pd.DataFrame(subtype_rows)
SUBTYPE_DF = (SUBTYPE_PAIRS_DF.groupby(["model", "subtype"])
              .agg(n=("gold_wins", "size"), accuracy=("gold_wins", "mean"), mean_margin=("margin", "mean"))
              .reset_index())
display(SUBTYPE_DF.sort_values(["model", "accuracy"]).round(3))
print("n küçük subtype satırlarını yalnız tanısal okuyun; inferential sonuç değildir.")

In [ ]:
heat = SUBTYPE_DF.pivot(index="subtype", columns="model", values="accuracy")
plt.figure(figsize=(max(7, 1.6 * len(RESULTS)), max(5, .45 * len(heat))))
sns.heatmap(heat, annot=True, fmt=".2f", cmap="RdYlGn", vmin=0, vmax=1)
plt.title("Gold'un hard negatifi geçme oranı"); plt.tight_layout(); plt.show()

## 10. Slice sonuçları

In [ ]:
slice_rows = []
for model_name, result in RESULTS.items():
    for metric in ["pairwise_hard_accuracy", "pairwise_morph_hard_accuracy", "pairwise_semantic_hard_accuracy", "recall@1"]:
        nested = slice_summary(result["per_query"], EVAL_ITEMS, metric=metric)
        for field, groups in nested.items():
            for value, stats in groups.items():
                slice_rows.append({"model": model_name, "metric": metric, "field": field, "value": value, **stats})
SLICE_DF = pd.DataFrame(slice_rows)
display(SLICE_DF.sort_values(["metric", "field", "model", "value"]).round(3))
print("n<5 dilimler yalnız hata bulma amaçlıdır; ayrı paper iddiası kurmayın.")

## 11. Hata analizi: model neyi birinci getirdi?

In [ ]:
item_by_id = {item["family_id"]: item for item in EVAL_ITEMS}
error_rows = []
for model_name, result in RESULTS.items():
    per_query = {row["query_id"]: row for row in result["per_query"]}
    for query_id, ranking in result["run"].items():
        item = item_by_id[query_id]
        candidates = {candidate["id"]: candidate for candidate in item["candidates"]}
        top = candidates[ranking[0]]
        row = per_query[query_id]
        error_rows.append({
            "model": model_name, "family_id": query_id, "correct@1": int(ranking[0] == item["gold_id"]),
            "gold_rank": row["rank"], "hard_rank": row["hard_rank"],
            "hardest_hard_margin": row["hardest_hard_margin"], "target_feature": item["target_feature"],
            "layer": item["layer"], "predicted_role": top["role"], "predicted_subtype": top["subtype"],
            "query": item["query"], "predicted_text": top["text"],
            "gold_text": candidates[item["gold_id"]]["text"],
        })
ERRORS_DF = pd.DataFrame(error_rows)
display(ERRORS_DF[ERRORS_DF["correct@1"] == 0].sort_values(["model", "gold_rank"]).reset_index(drop=True))

## 12. Paired model karşılaştırmaları

In [ ]:
def aligned_values(result, metric):
    return {row["query_id"]: float(row[metric]) for row in result["per_query"]}

comparison_rows, raw_p = [], {}
names = list(RESULTS)
for i, left_name in enumerate(names):
    for right_name in names[i + 1:]:
        for metric in ["pairwise_hard_accuracy", "recall@1", "ndcg@10"]:
            left_map, right_map = aligned_values(RESULTS[left_name], metric), aligned_values(RESULTS[right_name], metric)
            ids = sorted(set(left_map) & set(right_map))
            left, right = [left_map[x] for x in ids], [right_map[x] for x in ids]
            boot = paired_bootstrap(left, right, n_boot=N_BOOT, seed=SEED)
            p = approximate_randomization(left, right, n_iter=N_BOOT, seed=SEED)
            key = f"{left_name}__{right_name}__{metric}"
            raw_p[key] = p
            row = {"comparison": f"{left_name} - {right_name}", "metric": metric, "p_randomization": p, **boot}
            if metric == "recall@1":
                row.update({f"mcnemar_{k}": v for k, v in mcnemar(left, right).items()})
            comparison_rows.append(row)
adjusted = holm_adjust(raw_p) if raw_p else {}
for row in comparison_rows:
    left, right = row["comparison"].split(" - ")
    row["p_holm"] = adjusted[f"{left}__{right}__{row['metric']}"]
COMPARISONS_DF = pd.DataFrame(comparison_rows)
display(COMPARISONS_DF.round(3))
print("Effect size, güven aralığı ve Holm-düzeltilmiş p-değerlerini birlikte okuyun.")

## 13. Kritik sözcük / prefix-5 ablation

`prefix5`, gerçek Türkçe kök/lemma analizi değildir; ek bilgisini azaltan ucuz bir kontrol deneyidir.

In [ ]:
FOCUS = SUMMARY_DF["pairwise_hard_accuracy"].idxmax()
focus_spec = next(spec for spec in MODEL_SPECS if spec["name"] == FOCUS)
focus_model = load_model(focus_spec)
ablation_sets = {
    "original": EVAL_ITEMS,
    "critical_deleted": ablate_items(EVAL_ITEMS, "critical_deleted"),
    "prefix5": ablate_items(EVAL_ITEMS, "prefix5"),
}
ABLATION_RESULTS = {}
for ablation_name, items in ablation_sets.items():
    ABLATION_RESULTS[ablation_name], _ = score_with_backoff(focus_model, items, focus_spec)
del focus_model; gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
ABLATION_DF = pd.DataFrame({name: result["summary"] for name, result in ABLATION_RESULTS.items()}).T
ablation_metrics = ["pairwise_hard_accuracy", "contrast_consistency", "recall@1", "mrr@10", "ndcg@10"]
display(ABLATION_DF[ablation_metrics].round(3))
print("focus model:", FOCUS)

## 14. Full-corpus retrieval

Bu katman **her zaman** ortak corpus sıralamasını üretir:

- Final için 500 sealed query, bütün 5.500 aday içinde aranır.
- Her query için tasarım gereği tek gold vardır; diğer family'ler farklı semantic frame taşır.
- Cross-family exact/fuzzy duplicate ve frame kontrollerinden geçen diğer belgeler nonrelevant kabul edilir.
- `Recall@1/3/10/50`, `MRR@10` ve `nDCG@10` bütün corpus sıralaması üzerinde hesaplanır.

In [ ]:
# Full-corpus retrieval: her sealed query bütün 5.500 belgeyi sıralar.
full_corpus = {}
full_per_query_frames = []
for model_name, result in RESULTS.items():
    if "full_run" not in result:
        raise RuntimeError(f"{model_name} cache'i full_run içermiyor. Cache'i silip modeli yeniden çalıştırın.")
    summary, rows = evaluate_run(closed_qrels(EVAL_ITEMS), result["full_run"], recall_ks=FULL_CORPUS_RECALL_KS)
    full_corpus[model_name] = summary
    frame = pd.DataFrame(rows)
    frame.insert(0, "model", model_name)
    full_per_query_frames.append(frame)

FULL_CORPUS_DF = pd.DataFrame(full_corpus).T
FULL_CORPUS_PER_QUERY_DF = pd.concat(full_per_query_frames, ignore_index=True)
display(FULL_CORPUS_DF[[
    "recall@1", "recall@3", "recall@10", "recall@50", "mrr@10", "ndcg@10"
]].sort_values("recall@10", ascending=False).round(3))
print("Full corpus: 500 query, 5.500 ortak belge, query başına tek gold.")

## 15. Sonuçları dışa aktar

In [ ]:
SUMMARY_DF.to_csv(OUTPUT_DIR / "encoder_summary.csv", index_label="model")
FULL_CORPUS_DF.to_csv(OUTPUT_DIR / "full_corpus_retrieval.csv", index_label="model")
FULL_CORPUS_PER_QUERY_DF.to_csv(OUTPUT_DIR / "full_corpus_retrieval_per_query.csv", index=False)
ARTIFACT_DF.to_csv(OUTPUT_DIR / "artifact_baselines.csv", index_label="baseline")
CI_DF.to_csv(OUTPUT_DIR / "bootstrap_ci.csv", index=False)
SUBTYPE_DF.to_csv(OUTPUT_DIR / "hard_subtype_results.csv", index=False)
SLICE_DF.to_csv(OUTPUT_DIR / "slice_results.csv", index=False)
ERRORS_DF.to_csv(OUTPUT_DIR / "error_analysis.csv", index=False)
COMPARISONS_DF.to_csv(OUTPUT_DIR / "paired_comparisons.csv", index=False)
ABLATION_DF.to_csv(OUTPUT_DIR / "ablations.csv", index_label="ablation")
pd.DataFrame(RUNTIME_ROWS).to_csv(OUTPUT_DIR / "runtime.csv", index=False)
(OUTPUT_DIR / "model_errors.json").write_text(json.dumps(MODEL_ERRORS, ensure_ascii=False, indent=2), encoding="utf-8")
metadata = {
    "dataset_mode": "frozen_100_dev_500_test", "dataset_file": str(DATA_FILE), "dataset_sha256": DATA_SHA256,
    "evaluation_code_sha256": EVAL_CODE_SHA256, "evaluation_api": EVALUATION_API_VERSION,
    "git_commit": GIT_COMMIT, "full_run_depth": FULL_RUN_DEPTH,
    "family_count": len(EVAL_ITEMS), "seed": SEED, "bootstrap_draws": N_BOOT,
    "model_specs": MODEL_SPECS, "notice": DATA_NOTICE,
}
(OUTPUT_DIR / "run_metadata.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
archive = shutil.make_archive(str(OUTPUT_DIR), "zip", root_dir=OUTPUT_DIR)
print("çıktı:", OUTPUT_DIR)
print("zip:", archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    pass

## 16. Final sonucu yorumlama sırası

1. Önce ucuz baseline'ların yüksek olup olmadığına bakın; yüksekse veri artefaktı vardır.
2. Encoder'larda önce `recall@1`, `pairwise_hard_accuracy` ve margin'leri okuyun.
3. Hangi hard subtype'ların sürekli kaybedildiğini inceleyin.
4. Kritik sözcük silinince skorun düşmesi morfolojik sinyale duyarlılıkla uyumludur; tek başına nedensellik kanıtı değildir.
5. Model ve ayar seçimlerini yalnız 100 development üzerinde yapın; 500 sealed test sonucunu
   CI + paired testlerle nihai sonuç olarak raporlayın.
6. `full_corpus_retrieval.csv` bütün 5.500 belge üzerindeki retrieval sonucudur; kontrollü tabloyla birlikte raporlayın.
7. Fine-tuning seed varyansı frozen encoder baseline'ına uygulanmaz; training aşamasında en az üç seed ayrıca raporlanmalı.